In [1]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

This notebook creates the growth allocation list from the local growth assignments. The path to the log is here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx" this should be updated anytime new growth is reported and used as a reference and read in here. Images and lat/lons are here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_Residential.docx", "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_NonResidential.docx".    

Process: took the input from our locals (name and date recorded in spreadsheet) and found them on the map. Drew and then digitized a polygon and GIS analyst returns a list of Census blocks with the percent they are covered by these polygons. In this notebook we'll take the total growth reported and allocate it by percent to these blocks.

In [2]:
df = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx", sheet_name="Residential", usecols=["ID", "Number of People"])

In [4]:
df_perc = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim/StudyArea_Censusblocksummary.xls")

So this is filtered down where if a block is less than 50% intersected with the polygon, we don't allocate anything to it (then normalize percentages so it still allocates 100% of the value). Logic being that these values are taken as truth and we don't want to disrupt what's appearing there for something with potentially lower certainty/a lower value. Also it's an art not a science. However, in the case that none of the values exceed 50%... if one has 2 blocks that meet 50% threshold those two are retained, if one has 3 or more and none meet the threshold then all are retained.

In [6]:
#Identify R developments (no intersection threshold)
df_perc_filtered = df_perc.loc[~df_perc['ID'].str.contains('NR', na=False)].copy()

#re-normalize percentages to sum to 100% per development
df_perc_filtered['PCT_INTERSECT_NORM'] = (df_perc_filtered['PCT_INTERSECT']/ df_perc_filtered.groupby('ID')['PCT_INTERSECT'].transform('sum')* 100)

#match employment totals from development dataframe
df_perc_filtered['HHPOP_TOTAL'] = df_perc_filtered['ID'].map(df.set_index('ID')['Number of People'])

# Allocate employment proportionally to each block
df_perc_filtered['HHPOP_PER_BLOCK'] = (df_perc_filtered['HHPOP_TOTAL']* (df_perc_filtered['PCT_INTERSECT_NORM'] / 100))

# Match employment totals
df_perc_filtered['HHPOP_TOTAL'] = df_perc_filtered['ID'].map(df.set_index('ID')['Number of People'])

# Allocate employment
df_perc_filtered['HHPOP_PER_BLOCK'] = (df_perc_filtered['HHPOP_TOTAL']* (df_perc_filtered['PCT_INTERSECT_NORM'] / 100))

In [7]:
df_perc_filtered.sort_values(by = 'PCT_BLOCK', ascending = True).head(10)

,OBJECTID,ID,GEOID,PCT_INTERSECT,PCT_BLOCK,SIZE_BLOCK,PCT_INTERSECT_NORM,HHPOP_TOTAL,HHPOP_PER_BLOCK
15,16,R4,210472013042007,0.470441,0.334215,20.062600,5.769972,28.846154,1.664415
16,17,R4,210472013042008,3.456994,1.061358,8.670219,42.400125,28.846154,12.230805
7,8,R6,210472013041021,11.097861,8.064471,9.817085,8.658445,3.076923,0.266414
12,13,R6,210472013041018,10.978965,9.012814,11.090342,8.565683,3.076923,0.263559
10,11,R6,210472013041016,10.987697,9.293599,11.426763,8.572496,3.076923,0.263769
11,12,R6,210472013041019,11.944320,9.920222,11.220337,9.318844,3.076923,0.286734
5,6,R6,210472013041011,21.396367,10.703483,6.758200,16.693241,3.076923,0.513638
9,10,R6,210472013041015,19.609700,10.752658,7.407827,15.299299,3.076923,0.470748
8,9,R6,210472013041014,21.304503,10.887932,6.904304,16.621569,3.076923,0.511433
6,7,R6,210472013041017,20.650657,12.460657,8.151790,16.111445,3.076923,0.495737


In [8]:
df_perc_filtered = df_perc_filtered.rename(columns = {'GEOID': 'GEO_ID'})
df_perc_filtered['GEO_ID'] = df_perc_filtered['GEO_ID'].astype(str)

In [9]:
df.head()

,ID,Number of People
0,R1,115.384615
1,R2,61.538462
2,R3,23.076923
3,R4,28.846154
4,R5,4.615385


In [11]:
allocation_check = df_perc_filtered.groupby('ID').agg(
    PCT_TOTAL=('PCT_INTERSECT_NORM', 'sum'),
    HHPOP_TOTAL=('HHPOP_TOTAL', 'first'),
    HHPOP_ALLOCATED=('HHPOP_PER_BLOCK', 'sum')
)

allocation_check['HHPOP_DIFF'] = (
    allocation_check['HHPOP_TOTAL'] - allocation_check['HHPOP_ALLOCATED']
)

print(allocation_check)

    PCT_TOTAL  HHPOP_TOTAL  HHPOP_ALLOCATED    HHPOP_DIFF
ID                                                       
R1      100.0   115.384615       115.384615  0.000000e+00
R2      100.0    61.538462        61.538462  0.000000e+00
R3      100.0    23.076923        23.076923  3.552714e-15
R4      100.0    28.846154        28.846154  0.000000e+00
R5      100.0     4.615385         4.615385  0.000000e+00
R6      100.0     3.076923         3.076923  0.000000e+00
R7      100.0    15.833333        15.833333  0.000000e+00
R8      100.0     6.250000         6.250000  0.000000e+00


In [12]:
#check for missing ID's - Hannah hadn't run this one yet
missing_ids = df.loc[
    ~df['ID'].isin(df_perc_filtered['ID'])
]

print(missing_ids)

   ID  Number of People
8  R9         16.666667


In [14]:
#sum by GEOID and we'll output to read into the allocation function for future

#agg by census block
local_growth = (df_perc_filtered.groupby('GEO_ID')['HHPOP_PER_BLOCK'].sum().to_dict())

#export to JSON
output_path = Path(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim\local_growth_residential.json")

with open(output_path, 'w') as f:
    json.dump(local_growth, f, indent=4)